In [ ]:
# ============================================================
# RELAÇÕES DE EXISTÊNCIA E DE RECORRÊNCIA DAS RESSONÂNCIAS (CORE-SHELL)
# UM CÓDIGO SÓ, UMA PASTA SÓ (PASTA logo abaixo).
#
# Os checkpoints são achados pelo nome dentro de PASTA (e subpastas):
#   *mix0.00*.npz -> contraste na velocidade (f = 0)
#   *mix0.50*.npz -> contraste misto        (f = 0,5)
#   *mix1.00*.npz -> contraste na densidade (f = 1)
#   qualquer outro *mixX.XX*.npz entra no modelo de existência com f.
#
# PARTE A, FORMA DA TESE
#   A1. existência (Eq. 4.18), por regime:
#         P_res = 1/(1 + exp(-g)),
#         g = (c0 + c3 d + c9 d^2) + ln a (c1 + c6 d + c4 ln a + c5 ln b) + ln b (c2 + c8 d + c7 ln b)
#   A2. recorrência ancorada (Eq. 4.19), velocidade e misto:
#         x_{n+1} = x_ref A(eta,d) + B(eta,d) x_n,  x_ref = sqrt(3/(m m_t)) = sqrt(3 a^(2-f))
#   A3. recorrência não ancorada: ln x_{n+1} quadrático em (ln x_n, eta, d)
#
# PARTE B, FORMA COM A FASE Phi = c_M/c_A + d c_M/c_B = a^-(1-f) + d b^-(1-f)
#   B1. existência com f, um modelo só para qualquer f (u = ln Phi):
#         g = c0 + c1 u + c2 u^2 + c3 ln a + c4 d + c5 (ln b)^2 + c6 ln a ln b + c7 f ln a + c8 d ln b + c9 d f
#   B2. regra física Phi >= pi (nenhum coeficiente ajustado)
#   B3. recorrência com Delta_W = pi/Phi, por família de multipolo l (nos polos, raízes de D_l):
#         x_{l,n+1} = x_{l,n} + S (Delta_W/pi) (z_{l-1,n+1} - z_{l-1,n}),  z = zeros de j_{l-1}
#   B4. recorrência ancorada em Delta_W, mesma forma da Eq. 4.19, nos picos de W_T:
#         x_{n+1} = Delta_W A(eta,d) + B(eta,d) x_n
#
# PARTE C, COMPARAÇÃO ENTRE AS FORMAS (tabelas com o melhor valor em negrito + gráficos), tudo em W_T
#   C1. existência: tese x modelo com f x regra Phi >= pi, nas mesmas curvas de teste, com acurácia,
#       precisão, sensibilidade, especificidade, F1, ROC AUC, log-loss, Brier, falsos positivos e negativos;
#   C2. recorrência nos picos consecutivos de W_T, nas mesmas curvas de teste: ancorada em x_ref (tese),
#       ancorada em Delta_W, não ancorada (tese) e passo S Delta_W (1 coef.), com erros da posição,
#       do passo x_{n+1} - x_n, da sequência gerada a partir do 1o pico e do nº de picos até x_M = 1.
#
# DADOS
#   ressonâncias: ressonancias_malha (máximos de W_T, proeminência >= 0,1 em log10), 1e-4 <= x_M <= 1;
#   existe ressonância quando a curva tem pelo menos um pico;
#   famílias da B3: raízes de D_l em "candidatos" (só as conferidas pelo princípio do argumento).
#
# TREINO / TESTE (por curva inteira, 20 % de teste)
#   A1, B1, B2: mesmas curvas de teste (semente 42), pra comparação ser justa;
#   A2, A3, B4, C2: separação por curva das transições de W_T (semente 42), a mesma nas quatro formas;
#   B3: separação por curva das famílias de polos (semente 42).
#
# SAÍDAS: PASTA/relacoes_existencia_recorrencia_final/
#   forma_da_tese/<regime>/  tabelas de métricas, gráficos e fórmula da existência
#   forma_da_tese/           tabelas de coeficientes (formato das Tabelas 2 e 3 da tese)
#   forma_com_phi/           existência com f, regra Phi >= pi, recorrência com Delta_W (polos)
#   forma_com_phi/<regime>/  recorrência ancorada em Delta_W (W_T), e os coeficientes A e B em forma_com_phi/
#   comparacao/              tabelas e gráficos da Parte C, e os números em CSV
#   (cada tabela em PNG, PDF e SVG; as de coeficientes também em CSV e .tex)
# ============================================================

import os
import re
import sys
import glob
import shutil
import subprocess
import importlib.util

# ============================================================
# PASTA (a única coisa a ajustar)
# ============================================================
PASTA = "/content/drive/MyDrive/coreshell_final/resultados_mix"

if PASTA.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

# dependências: scikit-learn e, no Colab, LaTeX para os rótulos
if importlib.util.find_spec("sklearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
if shutil.which("latex") is None and os.path.exists("/content"):
    print("Instalando LaTeX (só na primeira vez, alguns minutos)...")
    subprocess.run("apt-get update -qq && apt-get install -y -qq texlive texlive-latex-extra "
                   "texlive-fonts-recommended dvipng cm-super", shell=True, check=False,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.optimize import brentq
from scipy.special import spherical_jn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

USA_LATEX = shutil.which("latex") is not None
plt.rcParams.update({"text.usetex": USA_LATEX, "font.family": "sans-serif"})
VIRG = "{,}" if USA_LATEX else ","     # vírgula decimal ("{,}" evita o espaço extra do LaTeX)
PCT = r"\%" if USA_LATEX else "%"

OUTPUT_ROOT = os.path.join(PASTA, "relacoes_existencia_recorrencia_final")
DIR_TESE = os.path.join(OUTPUT_ROOT, "forma_da_tese")
DIR_PHI = os.path.join(OUTPUT_ROOT, "forma_com_phi")
for pasta in (DIR_TESE, DIR_PHI):
    os.makedirs(pasta, exist_ok=True)

X_MIN = 1e-4
X_MAX = 1.0
TEST_FRACTION = 0.20
RANDOM_SEED = 42
SEMENTE_PHI = 2026
N_BOOT = 300
LMAX_FAMILIAS = 10

CASES = {
    "velocity": {"f": 0.00, "recorrencia": True, "nome": "Contraste na velocidade"},
    "mixed": {"f": 0.50, "recorrencia": True, "nome": "Contraste misto"},
    "density": {"f": 1.00, "recorrencia": False, "nome": "Contraste na densidade"},
}


# ============================================================
# CHECKPOINTS PELO NOME
# ============================================================
def achar_checkpoints(pasta):
    """{f: arquivo} para todo *mixX.XX*.npz dentro da pasta (e subpastas)."""
    achados = {}
    for arq in sorted(glob.glob(os.path.join(pasta, "**", "*.npz"), recursive=True)):
        nome = os.path.basename(arq)
        if "_dados" in nome or "aleatorio" in nome or not nome.startswith("checkpoint"):
            continue
        m = re.search(r"mix([0-9]+\.[0-9]+)", nome)
        if m:
            f0 = round(float(m.group(1)), 2)
            achados.setdefault(f0, []).append(arq)
    # se houver mais de um arquivo pro mesmo f, prefere o "reparado" e depois o nome mais curto
    return {f0: sorted(v, key=lambda a: ("reparado" not in a, len(a), a))[0] for f0, v in achados.items()}


GRADES = achar_checkpoints(PASTA)
if not GRADES:
    raise FileNotFoundError(f"Nenhum checkpoint *mixX.XX*.npz em {PASTA}")
print("Checkpoints encontrados:")
for f0, arq in sorted(GRADES.items()):
    print(f"  f = {f0:.2f}: {os.path.relpath(arq, PASTA)}")

for regime, caso in CASES.items():
    caso["checkpoint"] = GRADES.get(round(caso["f"], 2))
    if caso["checkpoint"] is None:
        print(f"AVISO: sem checkpoint para {caso['nome']} (f = {caso['f']:.2f}); esse regime fica de fora.")


# ############################################################################
# FUNÇÕES: LEITURA, TRANSIÇÕES E AJUSTES DA FORMA DA TESE
# ############################################################################
# ============================================================
# FUNÇÕES DE APOIO
# ============================================================
def eta_from_alpha_beta(alpha, beta):
    alpha = np.asarray(alpha, dtype=float)
    beta = np.asarray(beta, dtype=float)
    return (alpha - beta) / (alpha + beta)


def sphere_reference_velocity(alpha, mix_factor):
    """x_ref = sqrt(3/(m*mt)) da esfera homogênea (sem casca).

    m  = c_M/c_A = alpha**(f - 1)
    mt = Z_M/Z_A = 1/alpha
    =>  x_ref = sqrt(3 * alpha**(2 - f))   (f = 0: sqrt(3)*alpha, como no caso velocity)
    """
    alpha = np.asarray(alpha, dtype=float)
    if np.any(~np.isfinite(alpha)) or np.any(alpha <= 0.0):
        raise ValueError("Alpha precisa ser finito e positivo.")
    m = alpha ** (mix_factor - 1.0)
    mt = 1.0 / alpha
    return np.sqrt(3.0 / (m * mt))


def fase_phi(alpha, beta, delta, f):
    """Phi = c_M/c_A + delta c_M/c_B = alpha^-(1-f) + delta beta^-(1-f)."""
    alpha = np.asarray(alpha, dtype=float)
    beta = np.asarray(beta, dtype=float)
    return alpha ** (-(1.0 - f)) + np.asarray(delta, dtype=float) * beta ** (-(1.0 - f))


def delta_W(alpha, beta, delta, f):
    """Espaçamento Delta_W = pi/Phi."""
    return np.pi / fase_phi(alpha, beta, delta, f)


def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[valid], y_pred[valid]

    if len(y_true) == 0:
        return {"N": 0, "r": np.nan, "R2": np.nan, "RMSE": np.nan, "MAE": np.nan, "MAPE_percent": np.nan}

    residuals = y_true - y_pred
    r_pearson = float(np.corrcoef(y_true, y_pred)[0, 1]) if len(y_true) >= 2 else np.nan

    ss_res = float(np.sum(residuals ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if not np.isclose(ss_tot, 0.0) else np.nan

    rmse = float(np.sqrt(np.mean(residuals ** 2)))
    mae = float(np.mean(np.abs(residuals)))

    positive_y = np.abs(y_true) > 1e-15
    mape = (
        float(100.0 * np.mean(np.abs(residuals[positive_y] / y_true[positive_y])))
        if np.any(positive_y) else np.nan
    )

    return {"N": int(len(y_true)), "r": r_pearson, "R2": r2, "RMSE": rmse, "MAE": mae, "MAPE_percent": mape}


# ============================================================
# LEITURA DO CHECKPOINT (formato coreshell_final) + TRANSIÇÕES
# ============================================================
def curva_valida(item):
    if item.get("status") != "ok":
        return False
    reparo = item.get("reparo_velocity")
    return not (isinstance(reparo, dict) and not reparo.get("ok", True))


def load_curves_from_checkpoint(checkpoint_path):
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint não encontrado:\n{checkpoint_path}")

    with np.load(checkpoint_path, allow_pickle=True) as data:
        curves = data["curves"].item()

    rows = []
    for chave, item in curves.items():
        alpha = float(item.get("alpha", np.nan))
        beta = float(item.get("beta", np.nan))
        delta = float(item.get("delta", np.nan))

        if not (
            np.isfinite(alpha) and alpha > 0.0
            and np.isfinite(beta) and beta > 0.0
            and np.isfinite(delta)
            and curva_valida(item)
        ):
            continue

        peak_tuples = []
        for pico in item.get("ressonancias_malha", []) or []:
            xM = float(pico.get("xM_peak", np.nan))
            if not (np.isfinite(xM) and X_MIN <= xM <= X_MAX):
                continue
            fwhm = float(pico.get("FWHM_energy", np.nan))
            q = float(pico.get("Q_energy", np.nan))
            peak_tuples.append((xM, fwhm, q))

        peak_tuples.sort(key=lambda p: p[0])
        x_peaks = [p[0] for p in peak_tuples]

        # raízes de D_l por multipolo (só existem nas curvas de risco, alpha < 1 ou beta < 1),
        # usadas pela recorrência com Delta_W; entram só as conferidas pelo princípio do argumento
        raizes = {}
        for cand in item.get("candidatos", []) or []:
            re_, l = cand.get("pole_real"), int(cand.get("l", -1))
            if re_ is None or not bool(cand.get("argument_audit_ok", False)):
                continue
            if X_MIN < float(re_) <= X_MAX and 0 <= l <= 10:
                raizes.setdefault(l, []).append(float(re_))

        rows.append({
            "curve_key": chave, "alpha": alpha, "beta": beta,
            "eta": float(eta_from_alpha_beta(alpha, beta)), "delta": delta,
            "n_peaks": len(x_peaks), "x_peaks": x_peaks, "peaks": peak_tuples,
            "raizes": {l: np.sort(v) for l, v in raizes.items()},
        })

    return pd.DataFrame(rows), len(curves)


def build_transitions(curves_df):
    rows = []
    for _, row in curves_df.iterrows():
        x_peaks = row["x_peaks"]
        if len(x_peaks) < 2:
            continue
        for index in range(len(x_peaks) - 1):
            rows.append({
                "curve_key": row["curve_key"], "alpha": row["alpha"],
                "beta": row["beta"], "eta": row["eta"], "delta": row["delta"],
                "xM_n": x_peaks[index], "xM_np1": x_peaks[index + 1],
            })
    return pd.DataFrame(rows)


def split_train_test_by_curve(transitions_df, test_fraction, seed):
    rng = np.random.default_rng(seed)
    unique_keys = np.array(transitions_df["curve_key"].unique().tolist(), dtype=object)
    rng.shuffle(unique_keys)

    n_test = max(1, int(round(len(unique_keys) * test_fraction)))
    test_keys = set(unique_keys[:n_test])
    train_keys = set(unique_keys[n_test:])

    train_df = transitions_df.loc[transitions_df["curve_key"].isin(train_keys)].copy()
    test_df = transitions_df.loc[transitions_df["curve_key"].isin(test_keys)].copy()
    return train_df, test_df, len(train_keys), len(test_keys)


def split_curves_train_test(curves_df, test_fraction, seed):
    rng = np.random.default_rng(seed)
    index = np.arange(len(curves_df))
    rng.shuffle(index)
    n_test = max(1, int(round(len(index) * test_fraction)))
    return curves_df.iloc[index[n_test:]].copy(), curves_df.iloc[index[:n_test]].copy()


# ============================================================
# RECORRÊNCIA ANCORADA: x_{n+1} = x_anc A(eta,delta) + B(eta,delta) x_n
#   forma da tese:   x_anc = x_ref = sqrt(3 alpha^(2-f))  (esfera homogênea)
#   âncora Delta_W:  x_anc = Delta_W = pi/Phi              (mesma forma, outra âncora)
# ============================================================
COEF_NAMES_ANCORADA = [
    "a0", "a_eta", "a_delta", "a_eta2", "a_eta_delta", "a_delta2",
    "b0", "b_eta", "b_delta", "b_eta2", "b_eta_delta", "b_delta2",
]


def build_design_matrix_ancorada(alpha, eta, delta, x_n, mix_factor, x_anc=None):
    alpha = np.asarray(alpha, dtype=float)
    eta = np.asarray(eta, dtype=float)
    delta = np.asarray(delta, dtype=float)
    x_n = np.asarray(x_n, dtype=float)

    x_ref = sphere_reference_velocity(alpha, mix_factor) if x_anc is None else np.asarray(x_anc, dtype=float)

    XA = np.column_stack([x_ref, eta * x_ref, delta * x_ref, eta**2 * x_ref, eta * delta * x_ref, delta**2 * x_ref])
    XB = np.column_stack([x_n, eta * x_n, delta * x_n, eta**2 * x_n, eta * delta * x_n, delta**2 * x_n])
    return np.column_stack([XA, XB])


# ============================================================
# RECORRÊNCIA NÃO ANCORADA (log(x_n+1) livre)
# ============================================================
COEF_NAMES_NAO_ANCORADA = [
    "c0", "c_log_xn", "c_eta", "c_delta",
    "c_log_xn2", "c_eta2", "c_delta2",
    "c_log_xn_eta", "c_log_xn_delta", "c_eta_delta",
]


def build_design_matrix_nao_ancorada(eta, delta, x_n):
    eta = np.asarray(eta, dtype=float)
    delta = np.asarray(delta, dtype=float)
    log_xn = np.log(np.asarray(x_n, dtype=float))

    return np.column_stack([
        np.ones_like(log_xn),
        log_xn, eta, delta,
        log_xn**2, eta**2, delta**2,
        log_xn * eta, log_xn * delta, eta * delta,
    ])


def ols_standard_errors(X, y, coefficients):
    residuals = y - X @ coefficients
    n, p = X.shape
    s2 = float(residuals @ residuals) / max(n - p, 1)
    cov = s2 * np.linalg.pinv(X.T @ X)
    return np.sqrt(np.clip(np.diag(cov), 0.0, None))


def ancora_do_tipo(tipo, df, mix_factor):
    """None = x_ref da tese; no tipo "ancorada_deltaW" a âncora é Delta_W da curva."""
    if tipo == "ancorada_deltaW":
        return delta_W(df["alpha"].to_numpy(dtype=float), df["beta"].to_numpy(dtype=float),
                       df["delta"].to_numpy(dtype=float), mix_factor)
    return None


def fit_recurrence(curves_df, label, test_fraction, seed, tipo, mix_factor):
    """tipo: "ancorada" (x_ref, Eq. 4.19), "ancorada_deltaW" (âncora Delta_W) ou "nao_ancorada"."""
    anchored = tipo != "nao_ancorada"
    transitions_df = build_transitions(curves_df)

    if len(transitions_df) < 20:
        raise RuntimeError(f"[{label}] poucas transições ({len(transitions_df)}) pra ajustar/testar.")

    train_df, test_df, n_train_curves, n_test_curves = split_train_test_by_curve(
        transitions_df, test_fraction, seed
    )

    if anchored:
        def build_X(df):
            return build_design_matrix_ancorada(
                df["alpha"].to_numpy(dtype=float), df["eta"].to_numpy(dtype=float),
                df["delta"].to_numpy(dtype=float), df["xM_n"].to_numpy(dtype=float),
                mix_factor, ancora_do_tipo(tipo, df, mix_factor),
            )

        X_train = build_X(train_df)
        y_train = train_df["xM_np1"].to_numpy(dtype=float)
        coefficients, _, rank, sv = np.linalg.lstsq(X_train, y_train, rcond=None)

        def predict(df):
            return build_X(df) @ coefficients

        coef_names = COEF_NAMES_ANCORADA

    else:
        def build_X(df):
            return build_design_matrix_nao_ancorada(
                df["eta"].to_numpy(dtype=float), df["delta"].to_numpy(dtype=float),
                df["xM_n"].to_numpy(dtype=float),
            )

        X_train = build_X(train_df)
        y_train = np.log(train_df["xM_np1"].to_numpy(dtype=float))
        coefficients, _, rank, sv = np.linalg.lstsq(X_train, y_train, rcond=None)

        def predict(df):
            return np.exp(build_X(df) @ coefficients)

        coef_names = COEF_NAMES_NAO_ANCORADA

    standard_errors = ols_standard_errors(X_train, y_train, coefficients)

    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df["xM_np1_pred"] = predict(train_df)
    test_df["xM_np1_pred"] = predict(test_df)

    metrics_train = calculate_metrics(train_df["xM_np1"], train_df["xM_np1_pred"])
    metrics_test = calculate_metrics(test_df["xM_np1"], test_df["xM_np1_pred"])
    condition_number = float(sv[0] / sv[-1]) if len(sv) > 1 and sv[-1] > 0.0 else np.nan

    return {
        "label": label, "anchored": anchored, "tipo": tipo,
        "transitions_df": transitions_df,
        "train_df": train_df, "test_df": test_df,
        "coefficients": coefficients, "standard_errors": standard_errors, "coef_names": coef_names,
        "rank": int(rank), "condition_number": condition_number,
        "n_curves_total": int(len(curves_df)),
        "n_curves_with_peaks": int((curves_df["n_peaks"] > 0).sum()),
        "total_peaks": int(curves_df["n_peaks"].sum()),
        "n_train_curves": n_train_curves, "n_test_curves": n_test_curves,
        "metrics_train": metrics_train, "metrics_test": metrics_test,
    }


# ============================================================
# CLASSIFICADOR DE EXISTÊNCIA DE RESSONÂNCIA (Eq. 4.18)
# ============================================================
CLF_METRIC_ORDER = ["N", "Positive_rate", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]

CLF_TERM_NAMES = [
    "1", "ln_alpha", "ln_beta", "delta", "ln_alpha2",
    "ln_alpha_ln_beta", "ln_alpha_delta", "ln_beta2",
    "ln_beta_delta", "delta2",
]


def classifier_terms(alpha, beta, delta):
    ln_alpha = np.log(np.asarray(alpha, dtype=float))
    ln_beta = np.log(np.asarray(beta, dtype=float))
    delta = np.asarray(delta, dtype=float)
    return np.array([
        np.ones_like(ln_alpha),
        ln_alpha,
        ln_beta,
        delta,
        ln_alpha**2,
        ln_alpha * ln_beta,
        ln_alpha * delta,
        ln_beta**2,
        ln_beta * delta,
        delta**2,
    ])


def extract_classifier_formula(model):
    scaler = model.named_steps["scale"]
    logistic = model.named_steps["clf"]

    scaled_coefficients = logistic.coef_[0].astype(float)
    effective_coefficients = scaled_coefficients / scaler.scale_
    effective_intercept = float(
        logistic.intercept_[0]
        - np.sum(scaled_coefficients * scaler.mean_ / scaler.scale_)
    )

    return np.concatenate([[effective_intercept], effective_coefficients])


def classifier_score(alpha, beta, delta, coefficients):
    return np.tensordot(np.asarray(coefficients, dtype=float), classifier_terms(alpha, beta, delta), axes=(0, 0))


def classifier_probability(alpha, beta, delta, coefficients):
    score = classifier_score(alpha, beta, delta, coefficients)
    return 1.0 / (1.0 + np.exp(-np.clip(score, -700.0, 700.0)))


def logistic_standard_errors(train_df, coefficients):
    X = classifier_terms(train_df["alpha"], train_df["beta"], train_df["delta"]).T
    p = classifier_probability(train_df["alpha"], train_df["beta"], train_df["delta"], coefficients)
    w = p * (1.0 - p)
    fisher = X.T @ (X * w[:, None])
    cov = np.linalg.pinv(fisher)
    return np.sqrt(np.clip(np.diag(cov), 0.0, None))


def fit_existence_classifier(curves_df, test_fraction, seed):
    df = curves_df.copy()
    df["log_alpha"] = np.log(df["alpha"].to_numpy(dtype=float))
    df["log_beta"] = np.log(df["beta"].to_numpy(dtype=float))
    df["has_resonance"] = (df["n_peaks"] > 0).astype(int)

    train_df, test_df = split_curves_train_test(df, test_fraction, seed)

    feature_cols = ["log_alpha", "log_beta", "delta"]
    X_train = train_df[feature_cols].to_numpy(dtype=float)
    y_train = train_df["has_resonance"].to_numpy(dtype=int)
    X_test = test_df[feature_cols].to_numpy(dtype=float)
    y_test = test_df["has_resonance"].to_numpy(dtype=int)

    model = Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000)),
    ])
    model.fit(X_train, y_train)

    def evaluate(X, y):
        proba = model.predict_proba(X)[:, 1]
        pred = (proba >= 0.5).astype(int)
        metrics = {
            "N": int(len(y)), "Positive_rate": float(np.mean(y)),
            "Accuracy": float(accuracy_score(y, pred)),
            "Precision": float(precision_score(y, pred, zero_division=0)),
            "Recall": float(recall_score(y, pred, zero_division=0)),
            "F1": float(f1_score(y, pred, zero_division=0)),
        }
        metrics["ROC_AUC"] = float(roc_auc_score(y, proba)) if len(np.unique(y)) == 2 else np.nan
        return metrics, proba, pred

    metrics_train, proba_train, pred_train = evaluate(X_train, y_train)
    metrics_test, proba_test, pred_test = evaluate(X_test, y_test)

    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df["probability_resonance"] = proba_train
    train_df["predicted_resonance"] = pred_train
    test_df["probability_resonance"] = proba_test
    test_df["predicted_resonance"] = pred_test

    effective_coefficients = extract_classifier_formula(model)

    return {
        "model": model,
        "feature_cols": feature_cols,
        "train_df": train_df,
        "test_df": test_df,
        "formula_coefficients": effective_coefficients,
        "standard_errors": logistic_standard_errors(train_df, effective_coefficients),
        "formula_term_names": CLF_TERM_NAMES,
        "metrics_train": metrics_train,
        "metrics_test": metrics_test,
    }


# ############################################################################
# FUNÇÕES: TABELAS E GRÁFICOS
# ############################################################################
# ============================================================
# FORMATAÇÃO (vírgula decimal e porcentagem conforme LaTeX disponível)
# ============================================================
def format_decimal_comma(value, decimals=None):
    value = float(value)
    text = f"{value:g}" if decimals is None else f"{value:.{decimals}f}"
    return text.replace(".", VIRG)


def latex_sci(value, sig=3):
    if not np.isfinite(value):
        return "--"
    value = float(value)
    if np.isclose(value, 0.0):
        return "0"
    exponent = int(np.floor(np.log10(abs(value))))
    mantissa = value / (10.0 ** exponent)
    decimals = max(sig - 1, 0)
    mantissa_text = format_decimal_comma(mantissa, decimals)
    return rf"${mantissa_text}\times10^{{{exponent}}}$"


def fmt_int(value):
    return "--" if not np.isfinite(value) else f"{int(round(value))}"


def fmt_r(value):
    return "--" if not np.isfinite(value) else format_decimal_comma(value, 4)


def fmt_mape(value):
    return "--" if not np.isfinite(value) else f"{format_decimal_comma(value, 2)}{PCT}"


def fmt_pct01(value):
    return "--" if not np.isfinite(value) else f"{format_decimal_comma(100.0 * float(value), 1)}{PCT}"


def fmt_pm(value, error):
    """Valor ± incerteza, com a incerteza em 2 algarismos significativos (como nas Tabelas 2 e 3 da tese)."""
    if not np.isfinite(value):
        return "--"
    if not (np.isfinite(error) and error > 0.0):
        return f"${format_decimal_comma(value, 4)}$"
    decimals = max(0, -int(np.floor(np.log10(error))) + 1)
    valor = f"{value:+.{decimals}f}".replace(".", VIRG)
    incerteza = f"{error:.{decimals}f}".replace(".", VIRG)
    return rf"${valor}\pm{incerteza}$"


FORMATTERS_BY_METRIC = {"N": fmt_int, "r": fmt_r, "R2": fmt_r, "RMSE": latex_sci, "MAE": latex_sci, "MAPE_percent": fmt_mape}

CLF_FORMATTERS = {
    "N": fmt_int, "Positive_rate": fmt_pct01, "Accuracy": fmt_r, "Precision": fmt_r,
    "Recall": fmt_r, "F1": fmt_r, "ROC_AUC": fmt_r,
}

METRIC_ORDER = ["N", "r", "R2", "RMSE", "MAE", "MAPE_percent"]

row_labels_recorrencia = ["N\n(transições)", "Pearson\n" + r"$r$", r"$R^2$", "RMSE", "MAE", "MAPE\n(" + PCT + ")"]

row_labels_classifier = [
    "N\n(curvas)", "Taxa de\npositivos", "Acurácia", "Precisão", "Sensibilidade", "F1", "ROC AUC",
]


def build_table_data_2col(metrics_train, metrics_test, metric_order, formatters):
    return [[formatters[m](metrics_train.get(m, np.nan)), formatters[m](metrics_test.get(m, np.nan))] for m in metric_order]


# ============================================================
# TABELAS COMPACTAS (mesma estética das tabelas da tese)
# ============================================================
TABLE_FIGSIZE = (5.4, 4.5)
TABLE_DPI = 600


def save_compact_table(filename, table_data, row_labels, columns=("Treino", "Teste"),
                       figsize=TABLE_FIGSIZE, first_col_width=0.42, header_label="Parâmetro",
                       fontsize=19):
    with plt.rc_context({
        "font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 30,
        "figure.facecolor": "white", "axes.facecolor": "white",
        "savefig.facecolor": "white", "savefig.edgecolor": "white",
    }):
        fig, ax = plt.subplots(figsize=figsize, facecolor="white")
        ax.axis("off")
        ax.set_facecolor("white")

        n_cols = len(columns)
        cell_text = [[row_labels[i], *table_data[i]] for i in range(len(row_labels))]

        table = ax.table(
            cellText=cell_text, colLabels=[header_label, *columns], cellLoc="center",
            colWidths=[first_col_width] + [(1.0 - first_col_width) / n_cols] * n_cols,
            bbox=[0.02, 0.02, 0.96, 0.96],
        )
        table.auto_set_font_size(False)
        table.set_fontsize(fontsize)

        n_rows = len(row_labels)
        for (row, col), cell in table.get_celld().items():
            cell.visible_edges = ""
            cell.set_edgecolor("black")
            cell.set_linewidth(0.0)
            cell.set_facecolor("white")
            cell.PAD = 0.01
            text = cell.get_text()
            text.set_color("black")
            text.set_fontweight("normal")
            text.set_va("center")
            text.set_linespacing(0.90)
            text.set_ha("left" if col == 0 else "center")
            text.set_fontsize(fontsize)

        for col in range(n_cols + 1):
            cell = table[(0, col)]
            cell.visible_edges = "TB"
            cell.set_linewidth(1.5)
            cell.get_text().set_fontsize(fontsize + 1)
        table[(0, 0)].get_text().set_fontweight("bold")

        for col in range(n_cols + 1):
            cell = table[(n_rows, col)]
            cell.visible_edges = "B"
            cell.set_linewidth(1.5)

        fig.subplots_adjust(left=0.005, right=0.995, bottom=0.005, top=0.995)
        fig.savefig(f"{filename}.png", dpi=TABLE_DPI, bbox_inches="tight", pad_inches=0.01, facecolor="white", edgecolor="white")
        fig.savefig(f"{filename}.pdf", bbox_inches="tight", pad_inches=0.01, facecolor="white", edgecolor="white")
        fig.savefig(f"{filename}.svg", bbox_inches="tight", pad_inches=0.01, facecolor="white", edgecolor="white")
        plt.show()
        plt.close(fig)


# ============================================================
# GRÁFICOS (mesma estética das figuras da tese)
# ============================================================
def plot_single(result, output_dir, filename_slug):
    with plt.rc_context({
        "font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 16,
        "figure.facecolor": "white", "axes.facecolor": "white",
        "savefig.facecolor": "white", "savefig.edgecolor": "white",
    }):
        fig, ax = plt.subplots(figsize=(6.5, 5.0))
        test_df = result["test_df"]

        ax.scatter(test_df["xM_n"], test_df["xM_np1"], s=36, facecolors="#D9D9D9",
                   edgecolors="#8A8A8A", linewidths=0.45, alpha=0.75, zorder=1, label="Valor observado")
        ax.scatter(test_df["xM_n"], test_df["xM_np1_pred"], s=28, facecolors="#0f223d",
                   edgecolors="black", linewidths=0.45, alpha=0.9, zorder=3, label="Valor previsto")

        lims = [min(test_df["xM_n"].min(), test_df["xM_np1"].min()), max(test_df["xM_n"].max(), test_df["xM_np1"].max())]
        ax.plot(lims, lims, "--", color="0.35", linewidth=1.15, zorder=0, label="1:1")

        m = result["metrics_test"]
        text = (
            f"$N={m['N']}$\n"
            f"$r={format_decimal_comma(m['r'], 4)}$\n"
            f"$R^2={format_decimal_comma(m['R2'], 4)}$\n"
            f"RMSE = {latex_sci(m['RMSE'])}\n"
            f"MAPE = {format_decimal_comma(m['MAPE_percent'], 2)}{PCT}"
        )
        ax.text(0.035, 0.965, text, transform=ax.transAxes, ha="left", va="top", fontsize=10.5,
                linespacing=1.05, bbox=dict(boxstyle="square,pad=0.28", facecolor="white", edgecolor="black", linewidth=1.0))

        ax.set_xlabel(r"$x_M(n)$", fontsize=18, labelpad=7)
        ax.set_ylabel(r"$x_M(n+1)$", fontsize=18, labelpad=7)
        formatter_abnt = FuncFormatter(lambda value, pos: format_decimal_comma(value, 2))
        ax.xaxis.set_major_formatter(formatter_abnt)
        ax.yaxis.set_major_formatter(formatter_abnt)
        ax.grid(False)
        ax.tick_params(direction="in", length=5.0, width=1.15, labelsize=15, pad=4)
        for spine in ax.spines.values():
            spine.set_linewidth(1.15)
            spine.set_edgecolor("black")

        legend = ax.legend(loc="lower right", fontsize=10.5, frameon=True, fancybox=False,
                           framealpha=1.0, handlelength=2.0, borderpad=0.35,
                           labelspacing=0.28, handletextpad=0.45, borderaxespad=0.55)
        legend.get_frame().set_edgecolor("black")
        legend.get_frame().set_linewidth(1.0)
        legend.get_frame().set_facecolor("white")

        fig.subplots_adjust(left=0.18, right=0.97, bottom=0.17, top=0.97)
        base = os.path.join(output_dir, f"grafico_{filename_slug}")
        fig.savefig(base + ".png", dpi=600, bbox_inches="tight", pad_inches=0.03, facecolor="white", edgecolor="white")
        fig.savefig(base + ".pdf", bbox_inches="tight", pad_inches=0.03, facecolor="white", edgecolor="white")
        fig.savefig(base + ".svg", bbox_inches="tight", pad_inches=0.03, facecolor="white", edgecolor="white")
        plt.show()
        plt.close(fig)

    return base


def plot_existence_classifier(clf_result, output_dir, filename_slug):
    with plt.rc_context({
        "font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 16,
        "figure.facecolor": "white", "axes.facecolor": "white",
        "savefig.facecolor": "white", "savefig.edgecolor": "white",
    }):
        test_df = clf_result["test_df"].copy()
        y_true = test_df["has_resonance"].to_numpy(dtype=float)
        probabilities = test_df["probability_resonance"].to_numpy(dtype=float)

        rng = np.random.default_rng(RANDOM_SEED)
        x_jitter = y_true + rng.uniform(-0.045, 0.045, size=len(y_true))

        fig, ax = plt.subplots(figsize=(6.5, 5.0))

        no_resonance = y_true == 0
        resonance = y_true == 1

        ax.scatter(x_jitter[no_resonance], probabilities[no_resonance], s=28,
                   facecolors="#D9D9D9", edgecolors="#8A8A8A", linewidths=0.45,
                   alpha=0.75, zorder=2, label="Sem ressonância")
        ax.scatter(x_jitter[resonance], probabilities[resonance], s=28,
                   facecolors="#0f223d", edgecolors="black", linewidths=0.45,
                   alpha=0.85, zorder=3, label="Com ressonância")

        ax.plot([0.0, 1.0], [0.0, 1.0], "--", color="0.35", linewidth=1.15, zorder=0, label="1:1")
        ax.axhline(0.5, color="black", linestyle=":", linewidth=1.15, zorder=1,
                   label=r"Limiar $P_{\mathrm{res}}=0{,}5$")

        metrics = clf_result["metrics_test"]
        text = (
            f"$N={metrics['N']}$\n"
            f"Acurácia = {format_decimal_comma(metrics['Accuracy'], 3)}\n"
            f"Precisão = {format_decimal_comma(metrics['Precision'], 3)}\n"
            f"Sensibilidade = {format_decimal_comma(metrics['Recall'], 3)}\n"
            f"$F_1$ = {format_decimal_comma(metrics['F1'], 3)}\n"
            f"ROC AUC = {format_decimal_comma(metrics['ROC_AUC'], 3)}"
        )
        ax.text(0.035, 0.965, text, transform=ax.transAxes, ha="left", va="top",
                fontsize=10.5, linespacing=1.05,
                bbox=dict(boxstyle="square,pad=0.28", facecolor="white", edgecolor="black", linewidth=1.0))

        ax.set_xlim(-0.12, 1.12)
        ax.set_ylim(-0.02, 1.02)
        ax.set_xticks([0.0, 1.0])
        ax.set_xticklabels(["0", "1"], fontsize=15)
        ax.set_yticks([0.0, 0.25, 0.50, 0.75, 1.00])
        ax.set_yticklabels(["0", f"0{VIRG}25", f"0{VIRG}50", f"0{VIRG}75", f"1{VIRG}00"], fontsize=15)
        ax.set_xlabel("Existência observada de ressonância", fontsize=18, labelpad=7)
        ax.set_ylabel(r"Probabilidade prevista $P_{\mathrm{res}}$", fontsize=18, labelpad=7)
        ax.grid(False)
        ax.tick_params(direction="in", length=5.0, width=1.15, labelsize=15, pad=4)
        for spine in ax.spines.values():
            spine.set_linewidth(1.15)
            spine.set_edgecolor("black")
        legend = ax.legend(loc="lower right", fontsize=10.5, frameon=True, fancybox=False,
                           framealpha=1.0, handlelength=2.0, borderpad=0.35,
                           labelspacing=0.28, handletextpad=0.45, borderaxespad=0.55)
        legend.get_frame().set_edgecolor("black")
        legend.get_frame().set_linewidth(1.0)
        legend.get_frame().set_facecolor("white")

        fig.subplots_adjust(left=0.18, right=0.97, bottom=0.17, top=0.97)

        base = os.path.join(output_dir, f"grafico_{filename_slug}")
        fig.savefig(base + ".png", dpi=600, bbox_inches="tight", pad_inches=0.03, facecolor="white", edgecolor="white")
        fig.savefig(base + ".pdf", bbox_inches="tight", pad_inches=0.03, facecolor="white", edgecolor="white")
        fig.savefig(base + ".svg", bbox_inches="tight", pad_inches=0.03, facecolor="white", edgecolor="white")
        plt.show()
        plt.close(fig)

    return base


# ============================================================
# FÓRMULA DO CLASSIFICADOR EM TEXTO (pra copiar em código ou no LaTeX)
# ============================================================
def build_classifier_formula_text(coefficients, errors):
    c = np.asarray(coefficients, dtype=float)
    s = np.asarray(errors, dtype=float)

    symbolic = (
        "P_res(alpha,beta,delta) = 1 / (1 + exp[-g(alpha,beta,delta)])\n"
        "g = c0 + c1 ln(alpha) + c2 ln(beta) + c3 delta "
        "+ c4 [ln(alpha)]^2 + c5 ln(alpha)ln(beta) "
        "+ c6 delta ln(alpha) + c7 [ln(beta)]^2 "
        "+ c8 delta ln(beta) + c9 delta^2\n"
        "Existe ressonancia quando P_res >= 0.5, equivalentemente g >= 0.\n"
    )

    numeric = "\nCOEFICIENTES (valor, erro padrao)\n" + "".join(
        f"c{i} = {c[i]:+.12e}  +- {s[i]:.3e}\n" for i in range(len(c))
    )

    latex = (
        "\nLATEX\n"
        r"\begin{aligned}" "\n"
        r"P_{\mathrm{res}}(\alpha,\beta,\delta)"
        r"&=\frac{1}{1+\exp[-g(\alpha,\beta,\delta)]},\\" "\n"
        r"g(\alpha,\beta,\delta)"
        f"&={c[0]:+.8e}"
        rf"{c[1]:+.8e}\ln\alpha"
        rf"{c[2]:+.8e}\ln\beta"
        rf"{c[3]:+.8e}\delta"
        rf"{c[4]:+.8e}(\ln\alpha)^2"
        rf"{c[5]:+.8e}\ln\alpha\ln\beta"
        rf"{c[6]:+.8e}\delta\ln\alpha"
        rf"{c[7]:+.8e}(\ln\beta)^2"
        rf"{c[8]:+.8e}\delta\ln\beta"
        rf"{c[9]:+.8e}\delta^2."
        "\n" r"\end{aligned}" "\n"
    )

    return symbolic + numeric + latex


# ############################################################################
# PARTE A: FORMA DA TESE (A1 existência, A2 ancorada, A3 não ancorada)
# ############################################################################
RESULTADOS = {}

for regime, caso in CASES.items():
    if caso["checkpoint"] is None:
        continue
    pasta = os.path.join(DIR_TESE, regime)
    os.makedirs(pasta, exist_ok=True)

    print("=" * 96)
    print(f"{caso['nome'].upper()} | f={caso['f']:.2f}")
    print("=" * 96)
    print(f"Checkpoint: {caso['checkpoint']}")

    curves_df, n_total = load_curves_from_checkpoint(caso["checkpoint"])
    print(f"Curvas no checkpoint: {n_total} | usadas: {len(curves_df)} | com ressonância: {int((curves_df['n_peaks'] > 0).sum())}")

    RESULTADOS[regime] = {"curves_df": curves_df}

    # ---------------- recorrência ----------------
    if caso["recorrencia"]:
        tipos = [("ancorada", "Ancorada (esfera, x_ref)", pasta),
                 ("nao_ancorada", "Não ancorada", pasta),
                 ("ancorada_deltaW", "Ancorada em Delta_W", os.path.join(DIR_PHI, regime))]
        for kind_slug, kind_label, pasta_saida in tipos:
            os.makedirs(pasta_saida, exist_ok=True)
            result = fit_recurrence(curves_df, kind_label, TEST_FRACTION, RANDOM_SEED, kind_slug, caso["f"])
            RESULTADOS[regime][kind_slug] = result

            print(f"\n--- Recorrência {kind_label} ---")
            print(f"Transições: {len(result['transitions_df'])} | curvas treino/teste: {result['n_train_curves']}/{result['n_test_curves']}")
            print(f"Rank: {result['rank']}/{len(result['coef_names'])} | número de condição: {result['condition_number']:.3e}")
            for name, value, error in zip(result["coef_names"], result["coefficients"], result["standard_errors"]):
                print(f"  {name:>14s} = {value:+.6e}  ± {error:.2e}")
            print(f"TREINO: {result['metrics_train']}")
            print(f"TESTE : {result['metrics_test']}")

            table_data = build_table_data_2col(result["metrics_train"], result["metrics_test"], METRIC_ORDER, FORMATTERS_BY_METRIC)
            save_compact_table(os.path.join(pasta_saida, f"tabela_recorrencia_{kind_slug}"), table_data, row_labels_recorrencia)
            plot_single(result, pasta_saida, f"recorrencia_{kind_slug}")

    # ---------------- existência ----------------
    clf = fit_existence_classifier(curves_df, TEST_FRACTION, RANDOM_SEED)
    RESULTADOS[regime]["existencia"] = clf

    print("\n--- Existência ---")
    for name, value, error in zip(clf["formula_term_names"], clf["formula_coefficients"], clf["standard_errors"]):
        print(f"  {name:>18s} = {value:+.6e}  ± {error:.2e}")
    print(f"TREINO: {clf['metrics_train']}")
    print(f"TESTE : {clf['metrics_test']}")

    with open(os.path.join(pasta, "formula_existencia.txt"), "w", encoding="utf-8") as fh:
        fh.write(build_classifier_formula_text(clf["formula_coefficients"], clf["standard_errors"]))

    clf_table_data = build_table_data_2col(clf["metrics_train"], clf["metrics_test"], CLF_METRIC_ORDER, CLF_FORMATTERS)
    save_compact_table(os.path.join(pasta, "tabela_existencia"), clf_table_data, row_labels_classifier)
    plot_existence_classifier(clf, pasta, "existencia")

    print(f"\nArquivos em: {pasta}\n")


# ############################################################################
# PARTE A: TABELAS DE COEFICIENTES
# ############################################################################
def salvar_tabela_coeficientes(base, linhas, colunas, valores, erros, figsize, first_col_width=0.16, fontsize=17):
    """linhas: rótulos das linhas; colunas: rótulos das colunas; valores/erros: matrizes (linhas x colunas)."""
    dados = [[fmt_pm(valores[i][j], erros[i][j]) for j in range(len(colunas))] for i in range(len(linhas))]
    save_compact_table(base, dados, linhas, columns=colunas, figsize=figsize,
                       first_col_width=first_col_width, header_label="Coef.", fontsize=fontsize)

    registros = []
    for i, linha in enumerate(linhas):
        for j, coluna in enumerate(colunas):
            registros.append({"coeficiente": linha.replace("$", ""), "coluna": coluna,
                              "valor": valores[i][j], "erro_padrao": erros[i][j]})
    pd.DataFrame(registros).to_csv(base + ".csv", index=False)

    with open(base + ".tex", "w", encoding="utf-8") as fh:
        fh.write("\\begin{tabular}{l" + "c" * len(colunas) + "}\n\\hline\n")
        fh.write("Coef. & " + " & ".join(c.replace("\n", " ") for c in colunas) + " \\\\\n\\hline\n")
        for i, linha in enumerate(linhas):
            fh.write(linha + " & " + " & ".join(dados[i]) + " \\\\\n")
        fh.write("\\hline\n\\end{tabular}\n")


# ---------------- existência: c0..c9 por regime (ordem da Tabela 2) ----------------
ORDEM_EXISTENCIA = ["density", "velocity", "mixed"]
regimes_exist = [r for r in ORDEM_EXISTENCIA if "existencia" in RESULTADOS.get(r, {})]

salvar_tabela_coeficientes(
    os.path.join(DIR_TESE, "tabela_coeficientes_existencia"),
    [f"$c_{{{i}}}$" for i in range(10)],
    [CASES[r]["nome"].replace("Contraste ", "Contraste\n") for r in regimes_exist],
    [[RESULTADOS[r]["existencia"]["formula_coefficients"][i] for r in regimes_exist] for i in range(10)],
    [[RESULTADOS[r]["existencia"]["standard_errors"][i] for r in regimes_exist] for i in range(10)],
    figsize=(7.6, 6.2),
)

# ---------------- recorrência ancorada: A e B por regime (arranjo da Tabela 3) ----------------
regimes_rec = [r for r in ("velocity", "mixed") if "ancorada" in RESULTADOS.get(r, {})]


def tabela_coeficientes_ancorada(chave, base):
    colunas, valores, erros = [], [[] for _ in range(6)], [[] for _ in range(6)]
    for r in regimes_rec:
        res = RESULTADOS[r][chave]
        curto = "Velocidade" if r == "velocity" else "Misto"
        for parte, desloc in (("A", 0), ("B", 6)):
            colunas.append(f"{curto}\n${parte}$")
            for i in range(6):
                valores[i].append(res["coefficients"][desloc + i])
                erros[i].append(res["standard_errors"][desloc + i])
    salvar_tabela_coeficientes(base, [f"${i}$" for i in range(6)], colunas, valores, erros,
                               figsize=(12.5, 4.8), first_col_width=0.08, fontsize=16)


if regimes_rec:
    tabela_coeficientes_ancorada("ancorada", os.path.join(DIR_TESE, "tabela_coeficientes_recorrencia_ancorada"))
    # mesma forma, âncora Delta_W (fica junto das relações com Phi)
    tabela_coeficientes_ancorada("ancorada_deltaW", os.path.join(DIR_PHI, "tabela_coeficientes_recorrencia_ancorada_deltaW"))

# ---------------- recorrência não ancorada ----------------
ROTULOS_NAO_ANCORADA = [
    r"$1$", r"$\ln x_n$", r"$\eta$", r"$\delta$", r"$(\ln x_n)^2$",
    r"$\eta^2$", r"$\delta^2$", r"$\eta\ln x_n$", r"$\delta\ln x_n$", r"$\eta\delta$",
]
salvar_tabela_coeficientes(
    os.path.join(DIR_TESE, "tabela_coeficientes_recorrencia_nao_ancorada"),
    ROTULOS_NAO_ANCORADA,
    [CASES[r]["nome"].replace("Contraste ", "Contraste\n") for r in regimes_rec],
    [[RESULTADOS[r]["nao_ancorada"]["coefficients"][i] for r in regimes_rec] for i in range(10)],
    [[RESULTADOS[r]["nao_ancorada"]["standard_errors"][i] for r in regimes_rec] for i in range(10)],
    figsize=(6.4, 6.2),
)

print(f"Tabelas de coeficientes em: {DIR_TESE}")


# ############################################################################
# PARTE B: FORMA COM Phi (B1 existência com f, B2 regra Phi >= pi, B3 recorrência com Delta_W)
# ############################################################################


# ============================================================
# RECORRÊNCIA COM Delta_W  (x_{l,n+1} = x_{l,n} + S Delta_W/pi (z_{l-1,n+1} - z_{l-1,n}))
# ============================================================
def zeros_j_lmenos1(l, n):
    """n primeiros zeros positivos de j_{l-1}; para l = 0, j_{-1}(x) = cos(x)/x."""
    if l == 0:
        return (np.arange(n) + 0.5) * np.pi
    fn = lambda t: spherical_jn(l - 1, t)
    t = np.linspace(0.5, (n + l + 2) * np.pi, (n + l + 2) * 60)
    v = fn(t)
    i = np.where(v[:-1] * v[1:] < 0)[0][:n]
    return np.array([brentq(fn, t[k], t[k + 1]) for k in i])


ZEROS = {l: zeros_j_lmenos1(l, 3000) for l in range(LMAX_FAMILIAS + 1)}


def familias(df, f):
    """(dW, l, raízes) de cada família com pelo menos duas raízes."""
    saida = []
    for _, c in df.iterrows():
        dW = float(delta_W(c["alpha"], c["beta"], c["delta"], f))
        for l, xs in c["raizes"].items():
            if len(xs) >= 2 and len(xs) <= len(ZEROS[l]):
                saida.append((dW, l, np.asarray(xs, dtype=float), c["curve_key"]))
    return saida


def S_das_familias(fams):
    """Um S por família (mínimos quadrados sem intercepto, a partir da primeira raiz)."""
    vals = []
    for dW, l, xs, _ in fams:
        dz = ZEROS[l][:len(xs)] - ZEROS[l][0]
        vals.append(((xs - xs[0]) @ dz) / (dz @ dz) * np.pi / dW)
    return np.array(vals)


def transicoes_previstas(fams, S):
    """Passo a passo: x_{n+1} previsto a partir de x_n observado."""
    obs, prev = [], []
    for dW, l, xs, _ in fams:
        dz = np.diff(ZEROS[l][:len(xs)])
        obs.append(xs[1:])
        prev.append(xs[:-1] + S * dW / np.pi * dz)
    if not obs:
        return np.array([]), np.array([])
    return np.concatenate(obs), np.concatenate(prev)


def desvio_mediano_posicoes(fams, S):
    """Mesma métrica do Untitled32: desvio relativo médio das posições (1a raiz real + recorrência), mediana por curva."""
    por_curva = {}
    for dW, l, xs, chave in fams:
        p = xs[0] + S * dW / np.pi * (ZEROS[l][:len(xs)] - ZEROS[l][0])
        por_curva.setdefault(chave, []).append(np.abs(p[1:] - xs[1:]) / xs[1:])
    desv = [np.concatenate(v).mean() for v in por_curva.values()]
    return 100.0 * float(np.median(desv)) if desv else np.nan


def ajustar_S(df, f, seed):
    rng = np.random.default_rng(seed)
    chaves = np.array(df["curve_key"].tolist(), dtype=object)
    rng_split = np.random.default_rng(RANDOM_SEED)
    rng_split.shuffle(chaves)
    n_test = max(1, int(round(len(chaves) * TEST_FRACTION)))
    chaves_teste = set(chaves[:n_test])

    fams = familias(df, f)
    treino = [x for x in fams if x[3] not in chaves_teste]
    teste = [x for x in fams if x[3] in chaves_teste]

    Sv = S_das_familias(treino)
    if len(Sv) == 0:
        return None
    S = float(np.median(Sv))
    S_err = float(np.std([np.median(rng.choice(Sv, len(Sv))) for _ in range(N_BOOT)])) if len(Sv) > 1 else np.nan

    y_tr, p_tr = transicoes_previstas(treino, S)
    y_te, p_te = transicoes_previstas(teste, S)
    return {
        "S": S, "S_err": S_err, "n_familias": len(Sv),
        "metrics_train": calculate_metrics(y_tr, p_tr),
        "metrics_test": calculate_metrics(y_te, p_te),
        "desvio_teste": desvio_mediano_posicoes(teste, S),
        "teste_xn": np.concatenate([xs[:-1] for _, _, xs, _ in teste]) if teste else np.array([]),
        "teste_obs": y_te, "teste_prev": p_te,
    }


def plot_recorrencia_phi(res, output_dir, filename_slug):
    """Mesmo gráfico das recorrências da tese, com x_M(n) e x_M(n+1) da família l."""
    test_df = pd.DataFrame({"xM_n": res["teste_xn"], "xM_np1": res["teste_obs"], "xM_np1_pred": res["teste_prev"]})
    return plot_single({"test_df": test_df, "metrics_test": res["metrics_test"]}, output_dir, filename_slug)


# ============================================================
# EXISTÊNCIA COM f  (u = ln Phi)  E REGRA Phi >= pi
# ============================================================
TERMOS_F = ["1", "u", "u2", "ln_alpha", "delta", "ln_beta2", "ln_alpha_ln_beta", "f_ln_alpha", "delta_ln_beta", "delta_f"]
TEX_F = [r"$1$", r"$u$", r"$u^2$", r"$\ln\alpha$", r"$\delta$", r"$(\ln\beta)^2$",
         r"$\ln\alpha\ln\beta$", r"$f\ln\alpha$", r"$\delta\ln\beta$", r"$\delta f$"]


def termos_f(a, b, d, f):
    la, lb = np.log(a), np.log(b)
    u = np.log(np.exp(-(1 - f) * la) + d * np.exp(-(1 - f) * lb))
    return np.column_stack([np.ones_like(la), u, u * u, la, d, lb * lb, la * lb, f * la, d * lb, d * f])


def ajustar_logistica(P, y):
    """Logística com C = 100 e incertezas pela informação de Fisher (como no Untitled32)."""
    mu, sd = P[:, 1:].mean(0), P[:, 1:].std(0)
    Z = (P[:, 1:] - mu) / sd
    mod = LogisticRegression(C=100.0, max_iter=20000).fit(Z, y)
    b, b0 = mod.coef_[0], mod.intercept_[0]
    coef = np.r_[b0 - np.sum(b * mu / sd), b / sd]
    Zc = np.column_stack([np.ones(len(Z)), Z])
    p = mod.predict_proba(Z)[:, 1]
    cov = np.linalg.inv(Zc.T @ (Zc * (p * (1 - p))[:, None]) + np.eye(P.shape[1]) / 100.0)
    T = np.zeros((P.shape[1], P.shape[1]))
    T[0, 0] = 1.0
    T[0, 1:] = -mu / sd
    T[1:, 1:] = np.diag(1.0 / sd)
    return coef, np.sqrt(np.diag(T @ cov @ T.T))


def metricas_classificacao(y, pred, score=None):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=int)
    m = {
        "N": int(len(y)), "Positive_rate": float(np.mean(y)) if len(y) else np.nan,
        "Accuracy": float(accuracy_score(y, pred)) if len(y) else np.nan,
        "Precision": float(precision_score(y, pred, zero_division=0)) if len(y) else np.nan,
        "Recall": float(recall_score(y, pred, zero_division=0)) if len(y) else np.nan,
        "F1": float(f1_score(y, pred, zero_division=0)) if len(y) else np.nan,
    }
    m["ROC_AUC"] = float(roc_auc_score(y, score)) if score is not None and len(np.unique(y)) == 2 else np.nan
    return m


# ============================================================
# TODAS AS GRADES DE f ACHADAS EM PASTA
# ============================================================
DADOS_F = {}
for f0, arq in GRADES.items():
    regime = next((r for r, c in CASES.items() if np.isclose(c["f"], f0)), None)
    if regime is not None and regime in RESULTADOS:
        df = RESULTADOS[regime]["curves_df"].copy()
    else:
        df, _ = load_curves_from_checkpoint(arq)
    df["f"] = f0
    df["has_resonance"] = (df["n_peaks"] > 0).astype(int)
    DADOS_F[f0] = df

print("\nGrades de f usadas na forma com Phi: " + ", ".join(f"{f:g}" for f in sorted(DADOS_F)))

pasta_phi = DIR_PHI

# ---------------- existência com f ----------------
# teste = as mesmas curvas de teste da forma da tese (f = 0; 0,5; 1), pra comparação ser justa;
# em outras grades de f, 20 % das curvas sorteadas
rng = np.random.default_rng(SEMENTE_PHI)
partes = []
for f0, df in DADOS_F.items():
    regime = next((r for r, c in CASES.items() if np.isclose(c["f"], f0)), None)
    if regime is not None and "existencia" in RESULTADOS.get(regime, {}):
        chaves_teste = set(RESULTADOS[regime]["existencia"]["test_df"]["curve_key"])
        teste = df["curve_key"].isin(chaves_teste).to_numpy()
    else:
        teste = rng.random(len(df)) < TEST_FRACTION
    partes.append(df.assign(teste=teste))
todas = pd.concat(partes, ignore_index=True)
treino_f, teste_f = todas.loc[~todas["teste"]], todas.loc[todas["teste"]]

X_tr = termos_f(treino_f["alpha"].to_numpy(float), treino_f["beta"].to_numpy(float),
                treino_f["delta"].to_numpy(float), treino_f["f"].to_numpy(float))
coef_f, erro_f = ajustar_logistica(X_tr, treino_f["has_resonance"].to_numpy(int))

print("\n" + "=" * 96 + "\nEXISTÊNCIA COM f (modelo único)\n" + "=" * 96)
print(f"treino {len(treino_f)} curvas | teste {len(teste_f)} curvas")
for i in range(10):
    aviso = "" if abs(coef_f[i]) > 2 * erro_f[i] else "   <- compatível com zero"
    print(f"  c{i} ({TERMOS_F[i]:>16s}) = {coef_f[i]:+.5g} ± {erro_f[i]:.2g}{aviso}")


def avaliar_existencia(df):
    a, b, d, f = (df[k].to_numpy(float) for k in ("alpha", "beta", "delta", "f"))
    y = df["has_resonance"].to_numpy(int)
    g = termos_f(a, b, d, f) @ coef_f
    m_f = metricas_classificacao(y, g >= 0, 1.0 / (1.0 + np.exp(-np.clip(g, -700, 700))))
    m_phi = metricas_classificacao(y, fase_phi(a, b, d, f) >= np.pi)
    return m_f, m_phi


COMPARACAO = []
for f0 in sorted(DADOS_F):
    sub = teste_f.loc[np.isclose(teste_f["f"], f0)]
    m_f, m_phi = avaliar_existencia(sub)
    linha = {"f": f0, "modelo_f": m_f, "regra_phi": m_phi}
    regime = next((r for r, c in CASES.items() if np.isclose(c["f"], f0)), None)
    if regime is not None and "existencia" in RESULTADOS.get(regime, {}):
        coef_tese = RESULTADOS[regime]["existencia"]["formula_coefficients"]
        g_tese = classifier_score(sub["alpha"], sub["beta"], sub["delta"], coef_tese)
        linha["tese"] = metricas_classificacao(sub["has_resonance"], g_tese >= 0, 1.0 / (1.0 + np.exp(-np.clip(g_tese, -700, 700))))
    COMPARACAO.append(linha)
    print(f"  f = {f0:.2f} (teste, n = {m_f['N']}): modelo com f: acurácia {100 * m_f['Accuracy']:.2f}% | "
          f"regra Phi >= pi: acurácia {100 * m_phi['Accuracy']:.2f}%"
          + (f" | forma da tese: acurácia {100 * linha['tese']['Accuracy']:.2f}%" if "tese" in linha else ""))

m_f_tr, _ = avaliar_existencia(treino_f)
m_f_te, m_phi_te = avaliar_existencia(teste_f)
save_compact_table(os.path.join(pasta_phi, "tabela_existencia_com_f"),
                   build_table_data_2col(m_f_tr, m_f_te, CLF_METRIC_ORDER, CLF_FORMATTERS), row_labels_classifier)
save_compact_table(os.path.join(pasta_phi, "tabela_existencia_regra_phi"),
                   build_table_data_2col(avaliar_existencia(treino_f)[1], m_phi_te, CLF_METRIC_ORDER, CLF_FORMATTERS),
                   row_labels_classifier)

salvar_tabela_coeficientes(os.path.join(pasta_phi, "tabela_coeficientes_existencia_com_f"),
                           [f"$c_{{{i}}}$  " + TEX_F[i] for i in range(10)], ["Valor"],
                           [[coef_f[i]] for i in range(10)], [[erro_f[i]] for i in range(10)],
                           figsize=(5.4, 6.2), first_col_width=0.45)

# tabela comparativa da existência (teste): forma da tese x modelo com f x regra Phi >= pi
nomes_f = [f"$f={format_decimal_comma(l['f'], 2)}$" for l in COMPARACAO]
linhas_comp = ["Forma da tese\n(acurácia)", "Modelo com $f$\n(acurácia)", r"Regra $\Phi\geq\pi$" + "\n(acurácia)",
               "Modelo com $f$\n(sensibilidade)", r"Regra $\Phi\geq\pi$" + "\n(sensibilidade)"]
dados_comp = [
    [fmt_r(l["tese"]["Accuracy"]) if "tese" in l else "--" for l in COMPARACAO],
    [fmt_r(l["modelo_f"]["Accuracy"]) for l in COMPARACAO],
    [fmt_r(l["regra_phi"]["Accuracy"]) for l in COMPARACAO],
    [fmt_r(l["modelo_f"]["Recall"]) for l in COMPARACAO],
    [fmt_r(l["regra_phi"]["Recall"]) for l in COMPARACAO],
]
save_compact_table(os.path.join(pasta_phi, "tabela_comparacao_existencia"), dados_comp, linhas_comp,
                   columns=nomes_f, figsize=(2.6 + 1.6 * len(nomes_f), 4.4), first_col_width=0.36)

# ---------------- recorrência com Delta_W ----------------
print("\n" + "=" * 96 + "\nRECORRÊNCIA COM Delta_W\n" + "=" * 96)
RESULTADOS_S = {}
for f0 in sorted(DADOS_F):
    res = ajustar_S(DADOS_F[f0], f0, SEMENTE_PHI)
    if res is None:
        print(f"  f = {f0:.2f}: nenhuma família com duas raízes ou mais")
        continue
    RESULTADOS_S[f0] = res
    print(f"  f = {f0:.2f}: S = {res['S']:.4f} ± {res['S_err']:.4f} ({res['n_familias']} famílias de treino) | "
          f"teste passo a passo: R2 = {res['metrics_test']['R2']:.4f}, MAPE = {res['metrics_test']['MAPE_percent']:.2f}% | "
          f"desvio mediano das posições {res['desvio_teste']:.2f}%")

    regime = next((r for r, c in CASES.items() if np.isclose(c["f"], f0)), None)
    slug = regime if regime is not None else f"f{f0:.2f}"
    if res["metrics_test"]["N"] > 0:
        save_compact_table(os.path.join(pasta_phi, f"tabela_recorrencia_deltaW_{slug}"),
                           build_table_data_2col(res["metrics_train"], res["metrics_test"], METRIC_ORDER, FORMATTERS_BY_METRIC),
                           row_labels_recorrencia)
        plot_recorrencia_phi(res, pasta_phi, f"recorrencia_deltaW_{slug}")

if RESULTADOS_S:
    fs = sorted(RESULTADOS_S)
    save_compact_table(
        os.path.join(pasta_phi, "tabela_S_recorrencia_deltaW"),
        [[fmt_pm(RESULTADOS_S[f0]["S"], RESULTADOS_S[f0]["S_err"]), fmt_int(RESULTADOS_S[f0]["n_familias"]),
          fmt_mape(RESULTADOS_S[f0]["desvio_teste"])] for f0 in fs],
        [f"${format_decimal_comma(f0, 2)}$" for f0 in fs],
        columns=["$S$", "Famílias", "Desvio\nmediano"], figsize=(6.4, 1.2 + 0.55 * len(fs)),
        first_col_width=0.16, header_label="$f$", fontsize=17,
    )
    pd.DataFrame([{"f": f0, "S": RESULTADOS_S[f0]["S"], "S_erro": RESULTADOS_S[f0]["S_err"],
                   "familias": RESULTADOS_S[f0]["n_familias"], "desvio_mediano_pct": RESULTADOS_S[f0]["desvio_teste"],
                   "R2_teste": RESULTADOS_S[f0]["metrics_test"]["R2"], "MAPE_teste": RESULTADOS_S[f0]["metrics_test"]["MAPE_percent"]}
                  for f0 in fs]).to_csv(os.path.join(pasta_phi, "tabela_S_recorrencia_deltaW.csv"), index=False)

print(f"\nArquivos em: {pasta_phi}")


# ############################################################################
# PARTE C: COMPARAÇÃO ENTRE A FORMA DA TESE E A FORMA COM Phi
#
#   C1. EXISTÊNCIA, nas mesmas curvas de teste de cada regime:
#       forma da tese (10 coef. por regime) x modelo com f (10 coef. no total) x regra Phi >= pi (0 coef.)
#       quantidades: acurácia, precisão, sensibilidade, especificidade, F1, ROC AUC, log-loss, Brier,
#       falsos positivos e falsos negativos.
#   C2. RECORRÊNCIA, nos picos consecutivos de W_T (o mesmo alvo da tese), nas mesmas curvas de treino
#       e de teste: ancorada em x_ref (Eq. 4.19), ancorada em Delta_W (mesma forma, outra âncora),
#       não ancorada e o passo constante S Delta_W (1 coeficiente, referência mínima).
#       quantidades, no teste:
#         passo a passo: R2, RMSE, MAE e MAPE da posição; erro relativo do PASSO (x_{n+1} - x_n), que é o
#                        que a recorrência realmente prevê (a posição sozinha dá R2 perto de 1 em qualquer
#                        modelo, porque x_{n+1} ~ x_n): mediana, p90 e MAPE;
#         sequência inteira: a partir do 1o pico, a recorrência gera os picos seguintes até x_M = 1;
#                        desvio relativo das posições (mediana e p90) e erro no número de picos.
#   Em cada linha das tabelas o melhor valor sai em negrito.
# ############################################################################
DIR_COMP = os.path.join(OUTPUT_ROOT, "comparacao")
os.makedirs(DIR_COMP, exist_ok=True)

COR_TESE = "#0f223d"
COR_TESE_2 = "#810000"
COR_PHI = "#526F92"
COR_REGRA = "#D9D9D9"


def texto_simples(t):
    """Rótulo de figura -> texto de console/CSV (sem $, barras e quebras de linha)."""
    t = t.replace(chr(10), " ").replace(PCT, "%").replace("$", "")
    for a, b in ((r"x_{\mathrm{ref}}", "x_ref"), (r"\,", " "), (r"\Phi", "Phi"), (r"\pi", "pi"), (r"\geq", ">="), (r"\Delta_W", "Delta_W"), ("{,}", ",")):
        t = t.replace(a, b)
    return t


def negrito(texto):
    if texto in ("--", ""):
        return texto
    corpo = texto[1:-1] if texto.startswith("$") and texto.endswith("$") else texto
    return r"$\mathbf{" + corpo + "}$"


def tabela_com_melhor(valores, formatos, sentido):
    """valores: linhas x colunas (números); sentido: +1 maior é melhor, -1 menor é melhor, 0 sem destaque."""
    saida = []
    for linha, fmt, s in zip(valores, formatos, sentido):
        arr = np.array([v if v is not None and np.isfinite(v) else np.nan for v in linha], dtype=float)
        textos = [fmt(v) if np.isfinite(v) else "--" for v in arr]
        if s != 0 and np.isfinite(arr).sum() >= 2:
            alvo = np.nanmax(arr) if s > 0 else np.nanmin(arr)
            textos = [negrito(t) if np.isfinite(v) and np.isclose(v, alvo, rtol=1e-12, atol=0.0) else t
                      for t, v in zip(textos, arr)]
        saida.append(textos)
    return saida


def fmt_sci_ou_dec(v):
    return latex_sci(v) if abs(v) < 1e-2 and v != 0 else f"${format_decimal_comma(v, 4)}$"


# ============================================================
# C1. EXISTÊNCIA
# ============================================================
def metricas_existencia(y, prob, pred):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=int)
    vp = int(np.sum((pred == 1) & (y == 1)))
    vn = int(np.sum((pred == 0) & (y == 0)))
    fp = int(np.sum((pred == 1) & (y == 0)))
    fn = int(np.sum((pred == 0) & (y == 1)))
    m = {
        "Acurácia": (vp + vn) / len(y),
        "Precisão": vp / (vp + fp) if vp + fp else np.nan,
        "Sensibilidade": vp / (vp + fn) if vp + fn else np.nan,
        "Especificidade": vn / (vn + fp) if vn + fp else np.nan,
        "F1": 2 * vp / (2 * vp + fp + fn) if vp + fp + fn else np.nan,
        "ROC AUC": np.nan, "Log-loss": np.nan, "Brier": np.nan,
        "Falsos positivos": fp, "Falsos negativos": fn,
    }
    if prob is not None:
        p = np.clip(np.asarray(prob, dtype=float), 1e-12, 1 - 1e-12)
        if len(np.unique(y)) == 2:
            m["ROC AUC"] = roc_auc_score(y, p)
        m["Log-loss"] = float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
        m["Brier"] = float(np.mean((p - y) ** 2))
    return m


LINHAS_EXIST = [("Acurácia", fmt_r, +1), ("Precisão", fmt_r, +1), ("Sensibilidade", fmt_r, +1),
                ("Especificidade", fmt_r, +1), ("F1", fmt_r, +1), ("ROC AUC", fmt_r, +1),
                ("Log-loss", fmt_r, -1), ("Brier", fmt_r, -1),
                ("Falsos positivos", fmt_int, -1), ("Falsos negativos", fmt_int, -1),
                ("Coeficientes", fmt_int, 0)]
MODELOS_EXIST = ["Forma da tese", "Modelo com $f$", r"Regra $\Phi\geq\pi$"]
COMP_EXIST = {}

print("\n" + "=" * 96 + "\nC1. COMPARAÇÃO DA EXISTÊNCIA (mesmas curvas de teste)\n" + "=" * 96)
for regime, caso in CASES.items():
    if "existencia" not in RESULTADOS.get(regime, {}):
        continue
    f0 = caso["f"]
    sub = teste_f.loc[np.isclose(teste_f["f"], f0)]
    a, b, d = (sub[k].to_numpy(float) for k in ("alpha", "beta", "delta"))
    y = sub["has_resonance"].to_numpy(int)

    p_tese = classifier_probability(a, b, d, RESULTADOS[regime]["existencia"]["formula_coefficients"])
    g_f = termos_f(a, b, d, np.full_like(a, f0)) @ coef_f
    p_f = 1.0 / (1.0 + np.exp(-np.clip(g_f, -700, 700)))
    regra = fase_phi(a, b, d, f0) >= np.pi

    modelos = {
        "Forma da tese": metricas_existencia(y, p_tese, p_tese >= 0.5),
        "Modelo com $f$": metricas_existencia(y, p_f, p_f >= 0.5),
        r"Regra $\Phi\geq\pi$": metricas_existencia(y, None, regra),
    }
    modelos["Forma da tese"]["Coeficientes"] = 10
    modelos["Modelo com $f$"]["Coeficientes"] = 10
    modelos[r"Regra $\Phi\geq\pi$"]["Coeficientes"] = 0
    COMP_EXIST[regime] = modelos

    valores = [[modelos[mo][nome] for mo in MODELOS_EXIST] for nome, _, _ in LINHAS_EXIST]
    dados = tabela_com_melhor(valores, [fm for _, fm, _ in LINHAS_EXIST], [s for _, _, s in LINHAS_EXIST])
    rotulos = [nome for nome, _, _ in LINHAS_EXIST]
    save_compact_table(os.path.join(DIR_COMP, f"tabela_comparacao_existencia_{regime}"), dados, rotulos,
                       columns=["Forma da\ntese", "Modelo\ncom $f$", "Regra\n" + r"$\Phi\geq\pi$"],
                       figsize=(7.2, 6.6), first_col_width=0.34, header_label=f"{caso['nome']}\n(n = {len(y)})",
                       fontsize=17)

    print(f"\n{caso['nome']} (teste, n = {len(y)}):")
    for nome, _, _ in LINHAS_EXIST:
        print(f"  {nome:>18s}: " + " | ".join(f"{texto_simples(mo)}: {modelos[mo][nome]:.4g}"
                                             if np.isfinite(modelos[mo][nome]) else f"{texto_simples(mo)}: --"
                                             for mo in MODELOS_EXIST))

pd.DataFrame([{"regime": r, "modelo": texto_simples(mo), **COMP_EXIST[r][mo]}
              for r in COMP_EXIST for mo in MODELOS_EXIST]).to_csv(os.path.join(DIR_COMP, "comparacao_existencia.csv"), index=False)


def grafico_existencia(comp, base):
    metricas = ["Acurácia", "Precisão", "Sensibilidade", "Especificidade", "F1"]
    regimes = list(comp)
    cores = {"Forma da tese": COR_TESE, "Modelo com $f$": COR_PHI, r"Regra $\Phi\geq\pi$": COR_REGRA}
    with plt.rc_context({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 15,
                         "figure.facecolor": "white", "axes.facecolor": "white"}):
        fig, axes = plt.subplots(1, len(regimes), figsize=(5.2 * len(regimes), 4.6), sharey=True)
        axes = np.atleast_1d(axes)
        largura = 0.26
        for ax, regime in zip(axes, regimes):
            xpos = np.arange(len(metricas))
            for k, mo in enumerate(MODELOS_EXIST):
                vals = [comp[regime][mo][m] for m in metricas]
                ax.bar(xpos + (k - 1) * largura, np.nan_to_num(vals), width=largura, color=cores[mo],
                       edgecolor="black", linewidth=0.8, label=mo, zorder=3)
            ax.set_xticks(xpos)
            ax.set_xticklabels(["Acurácia", "Precisão", "Sensib.", "Especif.", "F1"], fontsize=13)
            ax.set_ylim(0.0, 1.05)
            ax.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
            ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: format_decimal_comma(v, 2)))
            ax.tick_params(direction="in", length=5.0, width=1.15, labelsize=13, pad=4)
            for spine in ax.spines.values():
                spine.set_linewidth(1.15)
                spine.set_edgecolor("black")
            ax.set_title(CASES[regime]["nome"], fontsize=15, pad=8)
        axes[0].set_ylabel("Valor no conjunto de teste", fontsize=16)
        fig.tight_layout(rect=(0.0, 0.09, 1.0, 1.0))
        handles, labels = axes[0].get_legend_handles_labels()
        legenda = fig.legend(handles, labels, loc="lower center", ncol=len(labels), fontsize=13,
                             frameon=True, fancybox=False, framealpha=1.0, bbox_to_anchor=(0.5, 0.0))
        legenda.get_frame().set_edgecolor("black")
        for ext in ("png", "pdf", "svg"):
            fig.savefig(f"{base}.{ext}", dpi=600, bbox_inches="tight", pad_inches=0.03, facecolor="white")
        plt.show()
        plt.close(fig)


if COMP_EXIST:
    grafico_existencia(COMP_EXIST, os.path.join(DIR_COMP, "grafico_comparacao_existencia"))


# ============================================================
# C2. RECORRÊNCIA NOS PICOS DE W_T
#   alvo comum: picos consecutivos de W_T de cada curva (x_n -> x_{n+1}), o mesmo da tese;
#   mesmas curvas de treino e de teste nas quatro formas (separação por curva, semente 42).
# ============================================================
class ModeloRec:
    """x_n -> x_{n+1} nos picos de W_T.

    "ancorada"        : x_{n+1} = x_ref A(eta,delta) + B(eta,delta) x_n   (Eq. 4.19, 12 coef.)
    "ancorada_deltaW" : x_{n+1} = Delta_W A(eta,delta) + B(eta,delta) x_n (mesma forma, 12 coef.)
    "nao_ancorada"    : ln x_{n+1} quadrático em (ln x_n, eta, delta)      (10 coef.)
    "passo_deltaW"    : x_{n+1} = x_n + S Delta_W                          (1 coef.)
    """

    N_COEF = {"ancorada": 12, "ancorada_deltaW": 12, "nao_ancorada": 10, "passo_deltaW": 1}

    def __init__(self, coef, tipo, f):
        self.coef, self.tipo, self.f = np.atleast_1d(np.asarray(coef, dtype=float)), tipo, f

    def proximo(self, alpha, beta, eta, delta, x_n):
        alpha, beta, eta, delta, x_n = (np.atleast_1d(np.asarray(v, dtype=float)) for v in (alpha, beta, eta, delta, x_n))
        if self.tipo == "nao_ancorada":
            return np.exp(build_design_matrix_nao_ancorada(eta, delta, x_n) @ self.coef)
        if self.tipo == "passo_deltaW":
            return x_n + self.coef[0] * delta_W(alpha, beta, delta, self.f)
        x_anc = delta_W(alpha, beta, delta, self.f) if self.tipo == "ancorada_deltaW" else None
        return build_design_matrix_ancorada(alpha, eta, delta, x_n, self.f, x_anc) @ self.coef


def ajustar_passo_deltaW(trans, f):
    """S por mínimos quadrados sem intercepto: (x_{n+1} - x_n) = S Delta_W."""
    dW = delta_W(trans["alpha"].to_numpy(float), trans["beta"].to_numpy(float), trans["delta"].to_numpy(float), f)
    passo = trans["xM_np1"].to_numpy(float) - trans["xM_n"].to_numpy(float)
    S = float(passo @ dW / (dW @ dW))
    S_err = float(ols_standard_errors(dW[:, None], passo, np.array([S]))[0])
    return S, S_err


def curvas_de_teste(curves_df, chaves):
    """Curvas de teste com pelo menos dois picos de W_T (as que têm transições)."""
    saida = []
    for _, c in curves_df.loc[curves_df["curve_key"].isin(chaves)].iterrows():
        if len(c["x_peaks"]) >= 2:
            saida.append({"chave": c["curve_key"], "alpha": c["alpha"], "beta": c["beta"], "eta": c["eta"],
                          "delta": c["delta"], "xs": np.asarray(c["x_peaks"], dtype=float)})
    return saida


def sequencias(modelo, curvas):
    """A partir do 1o pico, a recorrência gera os picos seguintes até passar de X_MAX (vetorizado)."""
    if not curvas:
        return []
    alpha, beta, eta, delta = (np.array([c[k] for c in curvas]) for k in ("alpha", "beta", "eta", "delta"))
    limite = np.array([4 * len(c["xs"]) + 50 for c in curvas])
    seqs = [[float(c["xs"][0])] for c in curvas]
    atual = np.array([c["xs"][0] for c in curvas], dtype=float)
    ativo = np.ones(len(curvas), dtype=bool)
    while ativo.any():
        idx = np.flatnonzero(ativo)
        with np.errstate(all="ignore"):
            nxt = modelo.proximo(alpha[idx], beta[idx], eta[idx], delta[idx], atual[idx])
        ok = np.isfinite(nxt) & (nxt > atual[idx]) & (nxt <= X_MAX)
        for j in np.flatnonzero(ok):
            seqs[idx[j]].append(float(nxt[j]))
        atual[idx[ok]] = nxt[ok]
        ativo[idx[~ok]] = False
        ativo &= np.array([len(sq) < lim for sq, lim in zip(seqs, limite)])
    return [np.array(sq) for sq in seqs]


def metricas_recorrencia(modelo, trans, curvas):
    obs = trans["xM_np1"].to_numpy(float)
    x_n = trans["xM_n"].to_numpy(float)
    pred = modelo.proximo(trans["alpha"], trans["beta"], trans["eta"], trans["delta"], x_n)
    m = calculate_metrics(obs, pred)
    passo_obs, passo_pred = obs - x_n, pred - x_n
    ok = passo_obs > 0
    erro_passo = np.abs(passo_pred[ok] - passo_obs[ok]) / passo_obs[ok]
    m["passo_mediana"] = 100 * float(np.median(erro_passo)) if len(erro_passo) else np.nan
    m["passo_p90"] = 100 * float(np.percentile(erro_passo, 90)) if len(erro_passo) else np.nan
    m["passo_MAPE"] = 100 * float(np.mean(erro_passo)) if len(erro_passo) else np.nan

    desv, erro_n, exatas = [], [], []
    for c, seq in zip(curvas, sequencias(modelo, curvas)):
        xs = c["xs"]
        n = min(len(seq), len(xs))
        if n >= 2:
            desv.append(np.abs(seq[1:n] - xs[1:n]) / xs[1:n])
        erro_n.append(abs(len(seq) - len(xs)) / len(xs))
        exatas.append(len(seq) == len(xs))
    desv = np.concatenate(desv) if desv else np.array([])
    m["seq_mediana"] = 100 * float(np.median(desv)) if len(desv) else np.nan
    m["seq_p90"] = 100 * float(np.percentile(desv, 90)) if len(desv) else np.nan
    m["contagem_erro"] = 100 * float(np.mean(erro_n)) if erro_n else np.nan
    m["contagem_exata"] = 100 * float(np.mean(exatas)) if exatas else np.nan
    m["erros_passo"] = erro_passo
    m["n_coef"] = ModeloRec.N_COEF[modelo.tipo]
    return m


NOME_ANC_TESE = "Ancorada em\n$x_{\\mathrm{ref}}$ (tese)"
NOME_ANC_DW = "Ancorada em\n$\\Delta_W$"
NOME_NAO_ANC = "Não ancorada\n(tese)"
NOME_PASSO_DW = "Passo\n$S\\,\\Delta_W$"
MODELOS_REC = [NOME_ANC_TESE, NOME_ANC_DW, NOME_NAO_ANC, NOME_PASSO_DW]

LINHAS_REC = [
    ("N (transições)", "N", fmt_int, 0),
    (r"$R^2$ (posição)", "R2", fmt_r, +1),
    ("RMSE (posição)", "RMSE", latex_sci, -1),
    ("MAE (posição)", "MAE", latex_sci, -1),
    ("MAPE posição (" + PCT + ")", "MAPE_percent", lambda v: format_decimal_comma(v, 2), -1),
    ("Erro do passo,\nmediana (" + PCT + ")", "passo_mediana", lambda v: format_decimal_comma(v, 2), -1),
    ("Erro do passo,\np90 (" + PCT + ")", "passo_p90", lambda v: format_decimal_comma(v, 2), -1),
    ("Sequência: desvio\nmediano (" + PCT + ")", "seq_mediana", lambda v: format_decimal_comma(v, 2), -1),
    ("Sequência: desvio\np90 (" + PCT + ")", "seq_p90", lambda v: format_decimal_comma(v, 2), -1),
    ("Erro no nº de\npicos (" + PCT + ")", "contagem_erro", lambda v: format_decimal_comma(v, 2), -1),
    ("Curvas com nº\nexato (" + PCT + ")", "contagem_exata", lambda v: format_decimal_comma(v, 1), +1),
    ("Coeficientes", "n_coef", fmt_int, 0),
]

COMP_REC = {}
PASSO_DW = {}
print("\n" + "=" * 96 + "\nC2. COMPARAÇÃO DA RECORRÊNCIA (picos consecutivos de W_T, teste)\n" + "=" * 96)
for regime, caso in CASES.items():
    if not caso["recorrencia"] or "ancorada" not in RESULTADOS.get(regime, {}):
        continue
    f0 = caso["f"]
    base_res = RESULTADOS[regime]["ancorada"]
    tr, te = base_res["train_df"], base_res["test_df"]          # mesma separação nas quatro formas
    curvas_te = curvas_de_teste(RESULTADOS[regime]["curves_df"], set(te["curve_key"]))

    S, S_err = ajustar_passo_deltaW(tr, f0)
    PASSO_DW[regime] = (S, S_err)
    modelos = {
        NOME_ANC_TESE: ModeloRec(RESULTADOS[regime]["ancorada"]["coefficients"], "ancorada", f0),
        NOME_ANC_DW: ModeloRec(RESULTADOS[regime]["ancorada_deltaW"]["coefficients"], "ancorada_deltaW", f0),
        NOME_NAO_ANC: ModeloRec(RESULTADOS[regime]["nao_ancorada"]["coefficients"], "nao_ancorada", f0),
        NOME_PASSO_DW: ModeloRec([S], "passo_deltaW", f0),
    }
    res = {nome: metricas_recorrencia(mod, te, curvas_te) for nome, mod in modelos.items()}
    COMP_REC[regime] = res

    valores = [[res[n][chave] for n in MODELOS_REC] for _, chave, _, _ in LINHAS_REC]
    dados = tabela_com_melhor(valores, [fm for _, _, fm, _ in LINHAS_REC], [s for _, _, _, s in LINHAS_REC])
    save_compact_table(os.path.join(DIR_COMP, f"tabela_comparacao_recorrencia_{regime}"), dados,
                       [r for r, _, _, _ in LINHAS_REC], columns=MODELOS_REC, figsize=(11.0, 8.2),
                       first_col_width=0.26, header_label=caso["nome"], fontsize=15)

    print(f"\n{caso['nome']}: {len(curvas_te)} curvas de teste, {len(te)} transições | "
          f"passo S Delta_W: S = {S:.5f} ± {S_err:.1e}")
    for rot, chave, _, _ in LINHAS_REC:
        print(f"  {texto_simples(rot):>32s}: " + " | ".join(
            f"{texto_simples(n)}: {res[n][chave]:.4g}" for n in MODELOS_REC))

pd.DataFrame([{"regime": r, "modelo": texto_simples(n), **{k: v for k, v in COMP_REC[r][n].items() if k != "erros_passo"}}
              for r in COMP_REC for n in COMP_REC[r]]).to_csv(os.path.join(DIR_COMP, "comparacao_recorrencia.csv"), index=False)


def grafico_recorrencia(comp, base):
    """Distribuição acumulada do erro relativo do passo, uma curva por forma."""
    estilos = {
        NOME_ANC_TESE: dict(color=COR_TESE, linestyle="-"),
        NOME_ANC_DW: dict(color="black", linestyle="-."),
        NOME_NAO_ANC: dict(color=COR_TESE_2, linestyle="--"),
        NOME_PASSO_DW: dict(color="#8A8A8A", linestyle=":"),
    }
    regimes = list(comp)
    with plt.rc_context({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 15,
                         "figure.facecolor": "white", "axes.facecolor": "white"}):
        fig, axes = plt.subplots(1, len(regimes), figsize=(6.0 * len(regimes), 4.8), sharey=True)
        axes = np.atleast_1d(axes)
        for ax, regime in zip(axes, regimes):
            for nome, est in estilos.items():
                e = np.sort(comp[regime][nome]["erros_passo"]) * 100
                e = e[e > 0]
                if len(e):
                    ax.plot(e, np.arange(1, len(e) + 1) / len(e), linewidth=2.0,
                            label=nome.replace("\n", " "), **est)
            ax.set_xscale("log")
            ax.set_ylim(0.0, 1.02)
            ax.set_xlabel("Erro relativo do passo (" + PCT + ")", fontsize=16)
            ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: format_decimal_comma(v, 2)))
            ax.tick_params(direction="in", length=5.0, width=1.15, labelsize=13, pad=4, which="both")
            for spine in ax.spines.values():
                spine.set_linewidth(1.15)
                spine.set_edgecolor("black")
            ax.set_title(CASES[regime]["nome"], fontsize=15, pad=8)
        axes[0].set_ylabel("Fração das transições de teste", fontsize=16)
        fig.tight_layout(rect=(0.0, 0.10, 1.0, 1.0))
        handles, labels = axes[0].get_legend_handles_labels()
        legenda = fig.legend(handles, labels, loc="lower center", ncol=4, fontsize=13, frameon=True,
                             fancybox=False, framealpha=1.0, bbox_to_anchor=(0.5, 0.0))
        legenda.get_frame().set_edgecolor("black")
        for ext in ("png", "pdf", "svg"):
            fig.savefig(f"{base}.{ext}", dpi=600, bbox_inches="tight", pad_inches=0.03, facecolor="white")
        plt.show()
        plt.close(fig)


if COMP_REC:
    grafico_recorrencia(COMP_REC, os.path.join(DIR_COMP, "grafico_comparacao_recorrencia"))


# ============================================================
# RESUMO: QUEM É MELHOR EM QUÊ
# ============================================================
print("\n" + "=" * 96 + "\nRESUMO DA COMPARAÇÃO\n" + "=" * 96)
for regime, modelos in COMP_EXIST.items():
    melhor_acc = max(MODELOS_EXIST, key=lambda mo: modelos[mo]["Acurácia"])
    melhor_ll = min([mo for mo in MODELOS_EXIST if np.isfinite(modelos[mo]["Log-loss"])],
                    key=lambda mo: modelos[mo]["Log-loss"])
    print(f"Existência, {CASES[regime]['nome']}: maior acurácia = {texto_simples(melhor_acc)} "
          f"({100 * modelos[melhor_acc]['Acurácia']:.2f}%), menor log-loss = {texto_simples(melhor_ll)}")
for regime, res in COMP_REC.items():
    candidatos = MODELOS_REC
    melhor_passo = min(candidatos, key=lambda n: res[n]["passo_mediana"])
    melhor_seq = min(candidatos, key=lambda n: res[n]["seq_mediana"])
    melhor_n = min(candidatos, key=lambda n: res[n]["contagem_erro"])
    print(f"Recorrência, {CASES[regime]['nome']}: menor erro do passo = {texto_simples(melhor_passo)} "
          f"| menor desvio da sequência = {texto_simples(melhor_seq)} "
          f"| menor erro no nº de picos = {texto_simples(melhor_n)}")
print(f"\nComparação salva em: {DIR_COMP}")

print("\n" + "=" * 96 + f"\nTUDO SALVO EM: {OUTPUT_ROOT}\n" + "=" * 96)